In [1]:
import sys
import os

# Adds the current directory to the search path
sys.path.append(os.path.abspath(''))

try:
    from alert_system import StudentMonitor
    print("✅ Success: alert_system imported!")
except ImportError as e:
    print(f"❌ Error: {e}")

✅ Success: alert_system imported!


In [ ]:
import os
import pandas as pd
import joblib
from alert_system import StudentMonitor

# --- PRE-FLIGHT CHECK ---
# Ensure models exist in the local directory
model_names = ['restlessness', 'impulsivity', 'irritability', 'insomnia']
models_dict = {}

try:
    for name in model_names:
        path = f"models/{name}_rf.pkl"
        models_dict[name] = joblib.load(path)
    print("✅ Technical Setup: All models loaded.")
except Exception as e:
    print(f"❌ CRITICAL: Model loading failed. Path issue? {e}")

# --- EDGE CASE 1: THE "DEAD PHONE" (Missing Keys) ---
# Testing how the code handles a dictionary missing expected sensor keys.
print("\n--- Testing Edge Case 1: Missing Data Keys ---")
monitor_tech = StudentMonitor(uid="u_tech_test", models=models_dict)

try:
    # Student sends an empty dict (phone off/no sensors)
    res = monitor_tech.process_new_day("2026-04-01", {}) 
    print(f"SUCCESS: System handled empty dict. Risk Score: {res.get('suicide_risk_score')}")
except Exception as e:
    # If this fails, we need to add .get(key, 0) to our code
    print(f"⚠️ FAILURE: System crashed on missing keys: {e}")

# --- EDGE CASE 2: THE "COLD START" (Day 1) ---
# Testing if the Z-Score logic breaks when there is no history.
print("\n--- Testing Edge Case 2: Cold Start (Day 1) ---")
monitor_new = StudentMonitor(uid="u_new_user", models=models_dict)

# Inputting extreme data on Day 1
extreme_day = {'night_unlocks': 100, 'dark_hrs': 0, 'total_convo_min': 0}
res_new = monitor_new.process_new_day("2026-04-01", extreme_day)

print(f"Alert on Day 1? {res_new.get('elevated_risk')} (Expected: False)")
print(f"Z-Score on Day 1: {res_new.get('z_score')} (Expected: 0 or NaN)")

# --- EDGE CASE 3: THE "OUTLIER" (Physically Impossible Data) ---
# Testing if the model handles a 'Distance' of 10,000km or 1 million unlocks.
print("\n--- Testing Edge Case 3: Impossible Outliers ---")
res_outlier = monitor_tech.process_new_day("2026-04-02", {'night_unlocks': 999999})
print(f"Result for impossible data: Z-Score = {res_outlier.get('z_score')}")

c:\Users\adan\miniconda3\envs\suicide_risk_env\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\adan\miniconda3\envs\suicide_risk_env\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


✅ Technical Setup: All models loaded.

--- Testing Edge Case 1: Missing Data Keys ---
SUCCESS: System handled empty dict. Risk Score: 44.949080601059435

--- Testing Edge Case 2: Cold Start (Day 1) ---
Alert on Day 1? False (Expected: False)
Z-Score on Day 1: 0.0 (Expected: 0 or NaN)

--- Testing Edge Case 3: Impossible Outliers ---
Result for impossible data: Z-Score = 0.7071067811864252


## 🛡️ Technical Robustness Audit Results

| Scenario | Result | Score/Metric | Significance |
| :--- | :--- | :--- | :--- |
| **Missing Data** | ✅ PASS | Risk: 44.95 | System is resilient to sensor failure/missing keys. |
| **Cold Start** | ✅ PASS | Z-Score: 0.0 | No "Division by Zero" errors; safe for new users. |
| **Outlier Input** | ✅ PASS | Z-Score: 0.71 | Logic prevents single-day "sensor noise" from triggering alerts. |

### Technical Verification Summary
The `StudentMonitor` class demonstrated high stability. It successfully handled null inputs and extreme outliers without runtime exceptions. The decision to default Z-scores to `0.0` during the initial baseline period (Day 1) ensures high **specificity** for new users.